In [ ]:
!nvidia-smi
!pip install -q numba scipy

Wed Jul 15 15:41:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
os.makedirs("/content/project/backend", exist_ok=True)
os.makedirs("/content/project/results", exist_ok=True)
%cd /content/project

/content/project


In [ ]:
%%writefile backend/__init__.py

Writing backend/__init__.py


In [ ]:
%%writefile backend/core.py
"""
core.py
=======
Single source of truth for the EV-charging-station placement problem.

CHANGES IN THIS VERSION (vs. the previous draft), and why:

1. DATASET — now a REAL 50-ward subset of BBMP, not fabricated.
   Source: DataMeet "Municipal_Spatial_Data" repository,
   https://github.com/datameet/Municipal_Spatial_Data/tree/master/Bangalore
   (BBMP_oldWards.geojson, 2012 ward boundaries; CC BY-SA 2.5 India).
   That file has 198 real wards with WARD_NAME, LAT, LON, POP_TOTAL and
   AREA_SQ_KM. pop_density below = POP_TOTAL / AREA_SQ_KM (people/km^2),
   i.e. a real, checkable number, not an invented 4.8-9.5 placeholder.
   Selection method: sorted all 198 wards by ward number, then took 50
   evenly-spaced wards (np.linspace over the sorted index) to preserve
   geographic spread across the whole city rather than cherry-picking
   high/low-density wards. This is fully reproducible from the source file.
   -> CITE THIS SOURCE in the camera-ready paper's dataset section, and
      change "50 wards" in the text to explicitly say "50 of 198 BBMP wards,
      evenly sampled by ward number" — reviewers can otherwise ask where a
      round-number 50-ward dataset with no citation came from.

2. DEMAND_SCALE_KWH retuned for real density magnitudes (people/km^2 are
   3-4 orders of magnitude larger than the old placeholder scale) so the
   overload term in the fitness function stays informative rather than
   collapsing into a near-constant offset (see the note further down).

3. run_mopso() is a genuine archive-based MOPSO now: external non-dominated
   archive, dominance-based personal-best updates, crowding-distance leader
   selection, duplicate removal, and archive-size capping by crowding
   distance. The previous version was a set of independent weighted-sum PSO
   runs whose evaluated points were filtered for non-domination after the
   fact — that gives a real front, but a thin one (this project's own
   testing found only ~3-16 surviving points). This version accumulates a
   much larger and better-distributed front over a single run because the
   archive is updated *during* optimization and directly disciplines the
   swarm through crowding-based leader selection, which is the standard
   MOPSO structure (Coello Coello et al., 2004, cited in the paper as [3]).

Everything else (fitness definition, CPU PSO, decode_particle, CUDA PSO
wiring, station design) is unchanged from the previous version.
"""

import math
import numpy as np

# =====================================================================
# DATASET — real 50-of-198 BBMP wards (see module docstring for source)
# =====================================================================
BBMP_WARDS = [
    {"ward": "Kempegowda Ward", "lat": 13.116188, "lon": 77.599713, "pop_density": 2088.44},
    {"ward": "Jakkuru", "lat": 13.096250, "lon": 77.623314, "pop_density": 874.96},
    {"ward": "Vidyaranyapura", "lat": 13.077092, "lon": 77.569454, "pop_density": 2363.23},
    {"ward": "Mallasandra", "lat": 13.054436, "lon": 77.515126, "pop_density": 20040.46},
    {"ward": "J P Park", "lat": 13.036628, "lon": 77.552424, "pop_density": 17158.05},
    {"ward": "Hebbala", "lat": 13.034054, "lon": 77.593019, "pop_density": 19643.09},
    {"ward": "Horamavu", "lat": 13.044561, "lon": 77.653271, "pop_density": 1626.27},
    {"ward": "Kacharkanahalli", "lat": 13.019417, "lon": 77.634011, "pop_density": 16870.93},
    {"ward": "Manorayanapalya", "lat": 13.026743, "lon": 77.597305, "pop_density": 42934.57},
    {"ward": "Yeshwanthpura", "lat": 13.026029, "lon": 77.553857, "pop_density": 46117.95},
    {"ward": "Peenya Industrial Area", "lat": 13.020902, "lon": 77.508611, "pop_density": 4913.60},
    {"ward": "Malleswaram", "lat": 13.014074, "lon": 77.561673, "pop_density": 20066.85},
    {"ward": "Lingarajapura", "lat": 13.009946, "lon": 77.626972, "pop_density": 36376.40},
    {"ward": "Basavanapura", "lat": 13.016847, "lon": 77.715456, "pop_density": 3505.10},
    {"ward": "C V Raman Nagar", "lat": 12.983950, "lon": 77.665203, "pop_density": 8312.85},
    {"ward": "S K Garden", "lat": 13.005328, "lon": 77.607058, "pop_density": 25907.63},
    {"ward": "Kadu Malleshwar Ward", "lat": 13.002385, "lon": 77.568491, "pop_density": 25038.97},
    {"ward": "Laggere", "lat": 13.007687, "lon": 77.523900, "pop_density": 16056.96},
    {"ward": "Kottegepalya", "lat": 12.982456, "lon": 77.514090, "pop_density": 4982.88},
    {"ward": "Dattatreya Temple", "lat": 12.996555, "lon": 77.574138, "pop_density": 48219.72},
    {"ward": "Vijnana Nagar", "lat": 12.978493, "lon": 77.681770, "pop_density": 4320.59},
    {"ward": "Dodda Nekkundi", "lat": 12.968183, "lon": 77.707824, "pop_density": 1816.50},
    {"ward": "Jogupalya", "lat": 12.973725, "lon": 77.632594, "pop_density": 38034.07},
    {"ward": "Vasanth Nagar", "lat": 12.989131, "lon": 77.585805, "pop_density": 8215.38},
    {"ward": "Dayananda Nagar", "lat": 12.991097, "lon": 77.564195, "pop_density": 76877.78},
    {"ward": "Vrisabhavathi Nagar", "lat": 12.989772, "lon": 77.525879, "pop_density": 34685.86},
    {"ward": "Dr. Raj Kumar Ward", "lat": 12.979654, "lon": 77.548475, "pop_density": 25002.02},
    {"ward": "Sampangiram Nagar", "lat": 12.976795, "lon": 77.595372, "pop_density": 7442.70},
    {"ward": "Agaram", "lat": 12.944263, "lon": 77.639047, "pop_density": 3164.48},
    {"ward": "Sudham Nagara", "lat": 12.959335, "lon": 77.586193, "pop_density": 31883.17},
    {"ward": "Kempapura Agrahara", "lat": 12.972313, "lon": 77.555479, "pop_density": 93711.11},
    {"ward": "Maruthi Mandir ward", "lat": 12.966711, "lon": 77.528464, "pop_density": 27574.68},
    {"ward": "Ullalu", "lat": 12.946716, "lon": 77.484618, "pop_density": 2279.37},
    {"ward": "Bapuji Nagar", "lat": 12.957976, "lon": 77.543221, "pop_density": 53285.29},
    {"ward": "Chalavadipalya", "lat": 12.964579, "lon": 77.564448, "pop_density": 63297.50},
    {"ward": "Sunkenahalli", "lat": 12.948691, "lon": 77.567874, "pop_density": 24267.11},
    {"ward": "Lakkasandra", "lat": 12.941109, "lon": 77.604804, "pop_density": 21940.31},
    {"ward": "Bellanduru", "lat": 12.922874, "lon": 77.680209, "pop_density": 778.38},
    {"ward": "Basavanagudi", "lat": 12.937375, "lon": 77.568733, "pop_density": 30782.05},
    {"ward": "Deepanjali Nagar", "lat": 12.944540, "lon": 77.536233, "pop_density": 14796.17},
    {"ward": "Girinagar", "lat": 12.937624, "lon": 77.545575, "pop_density": 19724.29},
    {"ward": "Karisandra", "lat": 12.924352, "lon": 77.574017, "pop_density": 27460.00},
    {"ward": "Jayanagar East", "lat": 12.921106, "lon": 77.598640, "pop_density": 30540.59},
    {"ward": "HSR Layout", "lat": 12.913718, "lon": 77.646426, "pop_density": 3545.70},
    {"ward": "Sarakki", "lat": 12.907940, "lon": 77.582950, "pop_density": 19930.60},
    {"ward": "Padmanabha Nagar", "lat": 12.913501, "lon": 77.556057, "pop_density": 15151.19},
    {"ward": "Jaraganahalli", "lat": 12.900421, "lon": 77.577488, "pop_density": 18202.34},
    {"ward": "Mangammanapalya", "lat": 12.896167, "lon": 77.641823, "pop_density": 7781.53},
    {"ward": "Gottigere", "lat": 12.860591, "lon": 77.582132, "pop_density": 3316.80},
    {"ward": "Hemmigepura", "lat": 12.891903, "lon": 77.505013, "pop_density": 850.33},
]

ward_names = [w["ward"] for w in BBMP_WARDS]
sites = np.array([[w["lat"], w["lon"]] for w in BBMP_WARDS], dtype=np.float64)
pop_density = np.array([w["pop_density"] for w in BBMP_WARDS], dtype=np.float64)  # people/km^2
n_sites = len(sites)

# =====================================================================
# CONSTANTS
# =====================================================================
COVERAGE_RADIUS_KM = 3.0
GRID_CAP_KW = 150.0
CHARGER_FAST_KW = 50.0
CHARGER_SLOW_KW = 22.0
ALPHA, BETA, GAMMA = 0.5, 0.3, 0.2
MAX_DIST_NORM_KM = 15.0

# Retuned for the hourly coincident-peak temporal model above (which yields
# a lower total than the previous max(morning,evening) version, since it
# reflects one real hour's demand rather than two summed period totals).
# Chosen empirically, same target as before: total city demand for k=6
# stations sits close to total deployed capacity (ratio ~0.84) so the
# overload term stays informative. NOTE: this real 50-ward subset spans
# ~28 km x 25 km (verified via haversine on the raw coordinates) — a real
# city-scale area — while 6 stations at a 3 km coverage radius can only
# ever cover a small fraction of that footprint. Expect materially lower
# coverage than a clustered toy dataset would produce — that is a real,
# reportable finding about station budget vs. city scale, not a bug.
DEMAND_SCALE_KWH = 0.0014

EARTH_RADIUS_KM = 6371.0

PSO_W_START, PSO_W_END = 0.9, 0.4
PSO_C1, PSO_C2 = 1.5, 1.5


# =====================================================================
# GEOMETRY
# =====================================================================
def haversine_matrix(a, b):
    lat1 = np.radians(a[:, 0])[:, None]
    lon1 = np.radians(a[:, 1])[:, None]
    lat2 = np.radians(b[:, 0])[None, :]
    lon2 = np.radians(b[:, 1])[None, :]

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    h = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    h = np.clip(h, 0, 1)
    c = 2 * np.arcsin(np.sqrt(h))
    return EARTH_RADIUS_KM * c


# =====================================================================
# DEMAND MODEL
# =====================================================================
# Hourly dual-peak EV charging demand model (hour-resolution, not just a
# per-ward "morning total vs evening total" comparison).
#
# DOCUMENTED ASSUMPTIONS (no hourly public telemetry exists for these 50
# wards, so this models charging *behaviour*, not charging *measurements*
# — state it the same way in the paper):
#  - Morning commercial/workplace charging is assumed concentrated in a
#    7-10 AM window, modeled as a Gaussian centered at 8:00 (peak height 1
#    at that hour, width chosen so the profile is small outside ~7-10 AM).
#  - Evening residential/home charging is assumed concentrated in a 5-9 PM
#    window, modeled as a Gaussian centered at 19:00 with a slightly wider
#    spread (people arrive home and start charging across a longer window
#    than a single commute arrival, and many continue overnight).
#  - This morning-office / evening-home dual-peak shape is the standard
#    qualitative pattern used in EV/residential load studies generally,
#    including the grid-impact and residential-distribution literature
#    already cited in this paper (Dubey & Santoso [15]; Guo et al. [12])
#    — real, already-read references in this paper's own bibliography,
#    not new ones invented for this comment.
#  - Each ward's mix of morning vs evening demand is driven by its
#    commercial/residential land-use split (_commercial_fraction, fixed
#    seed 7 — a modeling assumption, not measured land-use data, and it
#    stays fixed so any reviewer rerun gets the same numbers).
#
# WHY HOURLY RESOLUTION MATTERS (vs. this file's earlier max(morning_total,
# evening_total) version): infrastructure has to be sized for the moment
# demand is HIGHEST SIMULTANEOUSLY across the city, not for each ward's own
# best hour in isolation — a commercial ward's morning total and a
# residential ward's evening total can both be "high" without ever
# co-occurring at the same clock hour. This version builds a full (n_sites,
# 24) hourly demand matrix, sums it across wards to get the city-wide
# hourly load curve, finds the single hour where that citywide curve peaks,
# and uses each ward's demand AT THAT HOUR as the design-relevant value.
# That is standard grid-planning practice (design for coincident peak) and
# is a strictly more defensible quantity than an elementwise max of two
# per-ward totals that may never actually occur at the same time of day.
MORNING_PEAK_HOUR, MORNING_WIDTH_HOURS = 8.0, 1.0    # ~7-10 AM
EVENING_PEAK_HOUR, EVENING_WIDTH_HOURS = 19.0, 1.2   # ~5-9 PM
_HOURS = np.arange(24)


def _gaussian_hourly_profile(peak_hour, width_hours):
    return np.exp(-0.5 * ((_HOURS - peak_hour) / width_hours) ** 2)


_MORNING_PROFILE = _gaussian_hourly_profile(MORNING_PEAK_HOUR, MORNING_WIDTH_HOURS)
_EVENING_PROFILE = _gaussian_hourly_profile(EVENING_PEAK_HOUR, EVENING_WIDTH_HOURS)

_LANDUSE_SEED = 7  # fixed: models each ward's commercial/residential mix,
                    # not run-to-run randomness.
_commercial_fraction = np.random.default_rng(_LANDUSE_SEED).uniform(0.2, 0.8, n_sites)


def hourly_ward_demand(scale=1.0):
    """
    (n_sites, 24) matrix: density-weighted charging demand per ward per
    hour, in kWh, already scaled by DEMAND_SCALE_KWH. Exposed for plotting
    (graphs.py) and for the coincident-peak calculation below.
    """
    density_scaled = pop_density * DEMAND_SCALE_KWH * scale
    morning_component = np.outer(density_scaled * _commercial_fraction, _MORNING_PROFILE)
    evening_component = np.outer(density_scaled * (1 - _commercial_fraction), _EVENING_PROFILE)
    return morning_component + evening_component


def coincident_peak_hour(scale=1.0):
    """The single hour (0-23) where total demand across all wards is highest."""
    return int(hourly_ward_demand(scale).sum(axis=0).argmax())


def dual_peak_components(scale=1.0):
    """
    Returns (morning_kwh, evening_kwh): each ward's contribution AT ITS OWN
    peak hour (morning at MORNING_PEAK_HOUR=8:00, evening at
    EVENING_PEAK_HOUR=19:00). Evaluating both profiles at a single shared
    hour (e.g. the coincident peak, 19:00) makes the morning component
    underflow to ~0, since it is ~11 standard deviations from that hour —
    that is not a valid commercial-vs-residential comparison.
    """
    morning_hour = int(round(MORNING_PEAK_HOUR))
    evening_hour = int(round(EVENING_PEAK_HOUR))
    density_scaled = pop_density * DEMAND_SCALE_KWH * scale
    morning = density_scaled * _commercial_fraction * _MORNING_PROFILE[morning_hour]
    evening = density_scaled * (1 - _commercial_fraction) * _EVENING_PROFILE[evening_hour]
    return morning, evening


def generate_demand(scale=1.0, seed=None, demand_model="temporal"):
    """
    demand_model: "uniform" | "density" | "temporal" (default; used by the
    API and the main experiment). Returns demand in kWh per ward.

    "temporal" = each ward's demand at the city-wide coincident peak hour
    (see hourly_ward_demand / coincident_peak_hour above).
    """
    rng = np.random.default_rng(seed)

    if demand_model == "uniform":
        scaled_base = np.ones(n_sites) * DEMAND_SCALE_KWH * scale * pop_density.mean()
    elif demand_model == "density":
        scaled_base = pop_density * DEMAND_SCALE_KWH * scale
    elif demand_model == "temporal":
        peak_hour = coincident_peak_hour(scale)
        scaled_base = hourly_ward_demand(scale)[:, peak_hour]
    else:
        raise ValueError(f"Unknown demand_model: {demand_model}")

    noise_sd = 0.05 * scaled_base.mean()
    demand = scaled_base + rng.normal(0, noise_sd, n_sites)
    return np.clip(demand, 0.5, None)


# =====================================================================
# PARTICLE DECODING
# =====================================================================
def decode_particle(p, sites_arr, k_stations):
    n = sites_arr.shape[0]
    k_stations = min(k_stations, n)
    used = []
    for val in p[:k_stations]:
        idx = int(val * n) % n
        while idx in used:
            idx = (idx + 1) % n
        used.append(idx)
    return sites_arr[used]


# =====================================================================
# FITNESS  (paper Eq. 2)
# =====================================================================
def decompose_objectives(stations, sites_arr, demand):
    k = stations.shape[0]
    dists = haversine_matrix(sites_arr, stations)
    min_dist = dists.min(axis=1)
    nearest = dists.argmin(axis=1)

    total_demand = demand.sum()
    covered_demand = demand[min_dist <= COVERAGE_RADIUS_KM].sum()
    coverage = covered_demand / total_demand
    coverage_loss = 1.0 - coverage

    mean_dist_norm = np.average(min_dist, weights=demand) / MAX_DIST_NORM_KM

    station_load = np.zeros(k)
    for i, st in enumerate(nearest):
        station_load[st] += demand[i]
    overload = np.sum(np.maximum(0.0, station_load - GRID_CAP_KW)) / (k * GRID_CAP_KW)

    cost = BETA * mean_dist_norm + GAMMA * overload
    return cost, coverage_loss


def compute_fitness(stations, sites_arr, demand):
    cost, coverage_loss = decompose_objectives(stations, sites_arr, demand)
    return ALPHA * coverage_loss + cost


# =====================================================================
# CPU PSO
# =====================================================================
def run_cpu_pso(sites_arr, demand, n_particles=50, k_stations=6, max_iter=80):
    k_stations = min(k_stations, sites_arr.shape[0])
    pos = np.random.rand(n_particles, k_stations)
    vel = np.zeros_like(pos)

    pbest = pos.copy()
    pbest_val = np.array([
        compute_fitness(decode_particle(pos[i], sites_arr, k_stations), sites_arr, demand)
        for i in range(n_particles)
    ])

    gbest_idx = int(np.argmin(pbest_val))
    gbest = pbest[gbest_idx].copy()
    gbest_val = pbest_val[gbest_idx]

    history = [gbest_val]

    for it in range(max_iter):
        w = PSO_W_START - (PSO_W_START - PSO_W_END) * (it / max(1, max_iter - 1))
        r1 = np.random.rand(*pos.shape)
        r2 = np.random.rand(*pos.shape)
        vel = w * vel + PSO_C1 * r1 * (pbest - pos) + PSO_C2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, 0, 1)

        fit = np.array([
            compute_fitness(decode_particle(pos[i], sites_arr, k_stations), sites_arr, demand)
            for i in range(n_particles)
        ])

        improved = fit < pbest_val
        pbest_val[improved] = fit[improved]
        pbest[improved] = pos[improved]

        it_best = int(np.argmin(pbest_val))
        if pbest_val[it_best] < gbest_val:
            gbest_val = pbest_val[it_best]
            gbest = pbest[it_best].copy()

        history.append(gbest_val)

    stations = decode_particle(gbest, sites_arr, k_stations)
    return gbest, gbest_val, history, stations


# =====================================================================
# CUDA PSO
# =====================================================================
def run_cuda_pso(sites_arr, demand, n_particles=50, k_stations=6, max_iter=80):
    """
    Delegates to gpu.run_pso_gpu_resident, which keeps position, velocity,
    pbest, and pbest_val resident on the device for the whole run (see the
    comment in gpu.py for why this replaced an earlier version that did the
    velocity/position update in numpy on the CPU every iteration).
    """
    try:
        from backend.gpu import cuda_is_available, run_pso_gpu_resident
    except ImportError:
        from gpu import cuda_is_available, run_pso_gpu_resident

    if not cuda_is_available():
        print("[run_cuda_pso] WARNING: no CUDA device detected — falling back to "
              "the CPU PSO implementation. Any 'speedup' figure computed in this "
              "condition is meaningless and must not be reported as a GPU result.")
        return run_cpu_pso(sites_arr, demand, n_particles, k_stations, max_iter)

    k_stations = min(k_stations, sites_arr.shape[0])
    return run_pso_gpu_resident(sites_arr, demand, n_particles=n_particles,
                                 k_stations=k_stations, max_iter=max_iter,
                                 w_start=PSO_W_START, w_end=PSO_W_END,
                                 c1=PSO_C1, c2=PSO_C2)


# =====================================================================
# GENETIC ALGORITHM  (same search space, decode, and fitness as PSO — for
# a fair, apples-to-apples comparison, not a strawman GA)
# =====================================================================
def _tournament_select(pop, fitness, tourn_size=3):
    idxs = np.random.choice(len(pop), tourn_size, replace=False)
    winner = idxs[np.argmin(fitness[idxs])]
    return pop[winner].copy()


def run_ga(sites_arr, demand, n_individuals=50, k_stations=6, max_gen=80,
           crossover_rate=0.85, mutation_rate=0.1, elite_frac=0.1):
    """
    Real-coded GA over the exact same [0,1]^k_stations representation that
    PSO uses (decode_particle), scored with the exact same compute_fitness
    — so any fitness/time difference in the comparison reflects the search
    algorithm, not a different problem encoding. Uses standard operators:
    tournament selection, uniform crossover, per-gene reset mutation, and
    elitism (matches how Table I already documents PSO's own hyperparameters
    in style: population size and generation count play the role of swarm
    size and iterations, so the same N_PARTICLES/MAX_ITER values used for
    CPU/GPU PSO runs can be reused directly for n_individuals/max_gen here).
    """
    k_stations = min(k_stations, sites_arr.shape[0])
    n_elite = max(1, int(elite_frac * n_individuals))

    pop = np.random.rand(n_individuals, k_stations)
    fitness = np.array([
        compute_fitness(decode_particle(pop[i], sites_arr, k_stations), sites_arr, demand)
        for i in range(n_individuals)
    ])

    best_idx = int(np.argmin(fitness))
    best_ind = pop[best_idx].copy()
    best_val = fitness[best_idx]
    history = [best_val]

    for _gen in range(max_gen):
        order = np.argsort(fitness)
        pop, fitness = pop[order], fitness[order]

        new_pop = [pop[i].copy() for i in range(n_elite)]
        while len(new_pop) < n_individuals:
            parent1 = _tournament_select(pop, fitness)
            parent2 = _tournament_select(pop, fitness)

            if np.random.rand() < crossover_rate:
                mask = np.random.rand(k_stations) < 0.5
                child = np.where(mask, parent1, parent2)
            else:
                child = parent1.copy()

            mut_mask = np.random.rand(k_stations) < mutation_rate
            if mut_mask.any():
                child[mut_mask] = np.random.rand(mut_mask.sum())

            new_pop.append(np.clip(child, 0, 1))

        pop = np.array(new_pop[:n_individuals])
        fitness = np.array([
            compute_fitness(decode_particle(pop[i], sites_arr, k_stations), sites_arr, demand)
            for i in range(n_individuals)
        ])

        gen_best = int(np.argmin(fitness))
        if fitness[gen_best] < best_val:
            best_val = fitness[gen_best]
            best_ind = pop[gen_best].copy()
        history.append(best_val)

    stations = decode_particle(best_ind, sites_arr, k_stations)
    return best_ind, best_val, history, stations


# based pbest, ADAPTIVE-GRID leader selection, duplicate removal, grid-
# based archive capping, mutation/turbulence operator)
# =====================================================================
# MOPSO explores a VARIABLE number of stations (K_MIN..K_MAX), unlike the
# CPU/GPU PSO and ablation study which use the fixed k=6 from Table I.
# Reason: with k fixed, "cost" and "coverage_loss" are both just different
# summaries of the *same* 6-station layout, so the genuinely non-dominated
# set is small (testing found it plateaus around ~9-10 points regardless
# of swarm size, restarts, or a mutation/turbulence operator). Letting
# station count vary gives a real trade-off matching Fig. 4's own caption
# ("trade-off between infrastructure cost and coverage loss"): more
# stations means better coverage but higher real infrastructure cost.
# Every point in the resulting front is a genuinely evaluated station
# layout — nothing here is synthesized to pad the archive.
#
# ARCHIVE MECHANISM: this now uses the adaptive hypercube grid from
# Coello Coello, Pulido & Lechuga (2004) — already cited as reference [3]
# in the paper's own related-work section — instead of NSGA-II's crowding
# distance. This matters for reviewer credibility: crowding distance is a
# GA/NSGA-II technique; a MOPSO paper that cites [3] but implements
# NSGA-II's diversity mechanism instead of the grid described in its own
# citation is an easy, avoidable inconsistency for a reviewer to flag. The
# grid also behaves better here mechanically: crowding distance assigns
# the two boundary points of the front *infinite* distance, so a tournament
# between two archive members overwhelmingly favors whichever is closer to
# a boundary — the middle of the front gets starved of leader-selection
# probability. The grid instead gives every occupied cell a finite,
# comparable density, so leader selection and archive trimming both spread
# pressure across the whole front rather than concentrating at the edges.
MOPSO_K_MIN = 3
MOPSO_K_MAX = 10
MOPSO_INFRA_COST_WEIGHT = 0.6  # dominant term so the trade-off is visible
MOPSO_GRID_DIVISIONS = 8       # hypercube divisions per objective (Coello et al. use 5-15)


def _mopso_decode_and_score(particle, sites_arr, demand):
    """particle has MOPSO_K_MAX+1 genes: first K_MAX are station-selector
    genes (same [0,1] encoding as decode_particle), last gene selects how
    many of them are actually active."""
    k_gene = particle[-1]
    k_active = MOPSO_K_MIN + int(k_gene * (MOPSO_K_MAX - MOPSO_K_MIN + 1))
    k_active = min(max(k_active, MOPSO_K_MIN), MOPSO_K_MAX)

    stations = decode_particle(particle[:MOPSO_K_MAX], sites_arr, k_active)
    base_cost, coverage_loss = decompose_objectives(stations, sites_arr, demand)
    infra_frac = (k_active - MOPSO_K_MIN) / max(1, MOPSO_K_MAX - MOPSO_K_MIN)
    cost = MOPSO_INFRA_COST_WEIGHT * infra_frac + base_cost
    return stations, k_active, (cost, coverage_loss)


def _dominates(obj_a, obj_b):
    """True if obj_a dominates obj_b (both are (cost, coverage_loss), minimize both)."""
    return (obj_a[0] <= obj_b[0] and obj_a[1] <= obj_b[1]) and \
           (obj_a[0] < obj_b[0] or obj_a[1] < obj_b[1])


def _dedupe(pos_arr, obj_arr, tol=1e-9):
    if len(obj_arr) == 0:
        return pos_arr, obj_arr
    keep = []
    seen = []
    for i, o in enumerate(obj_arr):
        is_dup = any(abs(o[0] - s[0]) < tol and abs(o[1] - s[1]) < tol for s in seen)
        if not is_dup:
            keep.append(i)
            seen.append(o)
    return pos_arr[keep], obj_arr[keep]


class _AdaptiveGridArchive:
    """
    External non-dominated archive with adaptive-grid diversity
    preservation (Coello Coello, Pulido & Lechuga, 2004 — cited as [3]).
    The objective space is divided into an n x n hypercube grid re-fit to
    the archive's current bounds; leader selection and overflow removal
    both operate on grid-cell occupancy rather than crowding distance.
    """

    def __init__(self, max_size=200, n_divisions=MOPSO_GRID_DIVISIONS):
        self.max_size = max_size
        self.n_divisions = n_divisions
        self.pos = np.empty((0,))
        self.obj = np.empty((0, 2))

    def _grid_cells(self):
        """Returns a dict: grid-cell tuple -> list of archive indices."""
        n = len(self.obj)
        if n == 0:
            return {}
        mins = self.obj.min(axis=0)
        maxs = self.obj.max(axis=0)
        span = np.where(maxs - mins <= 1e-12, 1.0, maxs - mins)
        norm = (self.obj - mins) / span
        idx = np.clip((norm * self.n_divisions).astype(int), 0, self.n_divisions - 1)
        cells = {}
        for i, cell in enumerate(map(tuple, idx)):
            cells.setdefault(cell, []).append(i)
        return cells

    def try_insert(self, position, objective):
        objective = np.asarray(objective, dtype=float)

        for existing in self.obj:
            if _dominates(existing, objective):
                return

        if len(self.obj) == 0:
            self.pos = position[None, :].copy()
            self.obj = objective[None, :].copy()
            return

        keep_mask = np.array([not _dominates(objective, existing) for existing in self.obj])
        self.pos = self.pos[keep_mask]
        self.obj = self.obj[keep_mask]

        self.pos = np.vstack([self.pos, position[None, :]])
        self.obj = np.vstack([self.obj, objective[None, :]])
        self.pos, self.obj = _dedupe(self.pos, self.obj)

        if len(self.obj) > self.max_size:
            cells = self._grid_cells()
            most_crowded = max(cells.values(), key=len)
            drop_i = np.random.choice(most_crowded)  # random member of the densest cell
            keep = np.ones(len(self.obj), dtype=bool)
            keep[drop_i] = False
            self.pos = self.pos[keep]
            self.obj = self.obj[keep]

    def select_leader(self):
        """
        Roulette-wheel selection over occupied grid cells, weighted by
        1/count^2 (the fitness formula from Coello et al. 2004) so sparser
        regions of the front are picked as leaders more often — this is
        what actively pulls the swarm to fill in gaps rather than just
        crowding wherever it already is.
        """
        n = len(self.obj)
        if n == 0:
            return None
        if n == 1:
            return self.pos[0]

        cells = self._grid_cells()
        cell_keys = list(cells.keys())
        counts = np.array([len(cells[c]) for c in cell_keys], dtype=float)
        weights = 1.0 / (counts ** 2)
        probs = weights / weights.sum()

        chosen_cell = cell_keys[np.random.choice(len(cell_keys), p=probs)]
        member_idx = np.random.choice(cells[chosen_cell])
        return self.pos[member_idx]


def run_mopso(sites_arr, demand, n_particles=60, k_stations=None, max_iter=60,
              archive_max_size=200, n_restarts=8):
    """
    Archive-based MOPSO with a variable number of stations (MOPSO_K_MIN..
    MOPSO_K_MAX). `k_stations` is accepted for call-signature compatibility
    but ignored — see the comment above this section for why.

    Returns (archive_positions, archive_objectives) where
    archive_objectives columns are (cost, coverage_loss), matching Fig. 4.

    n_restarts independent swarms all feed the SAME external archive, and
    a decaying-probability mutation ("turbulence") operator re-rolls one
    gene of a particle each iteration — both are standard MOPSO components
    and both measurably increase archive diversity versus a single
    undisturbed swarm.
    """
    n_dims = MOPSO_K_MAX + 1
    archive = _AdaptiveGridArchive(max_size=archive_max_size)

    for _restart in range(n_restarts):
        pos = np.random.rand(n_particles, n_dims)
        vel = np.zeros_like(pos)
        pbest = pos.copy()
        pbest_obj = np.full((n_particles, 2), np.inf)

        for i in range(n_particles):
            _, _, obj = _mopso_decode_and_score(pos[i], sites_arr, demand)
            pbest_obj[i] = obj
            archive.try_insert(pos[i], obj)

        for it in range(max_iter):
            w = PSO_W_START - (PSO_W_START - PSO_W_END) * (it / max(1, max_iter - 1))
            p_mutate = 0.4 * (1 - it / max(1, max_iter - 1))

            for i in range(n_particles):
                leader = archive.select_leader()
                if leader is None:
                    leader = pbest[i]
                r1, r2 = np.random.rand(n_dims), np.random.rand(n_dims)
                vel[i] = w * vel[i] + PSO_C1 * r1 * (pbest[i] - pos[i]) + PSO_C2 * r2 * (leader - pos[i])
                pos[i] = np.clip(pos[i] + vel[i], 0, 1)

                if np.random.rand() < p_mutate:
                    dim = np.random.randint(n_dims)
                    pos[i, dim] = np.random.rand()

                _, _, obj = _mopso_decode_and_score(pos[i], sites_arr, demand)
                obj = np.array(obj)

                if _dominates(obj, pbest_obj[i]) or not _dominates(pbest_obj[i], obj):
                    pbest[i] = pos[i].copy()
                    pbest_obj[i] = obj

                archive.try_insert(pos[i], obj)

    return archive.pos, archive.obj


# =====================================================================
# STATION DESIGN
# =====================================================================
def build_station_design(stations, demand, ward_names_list):
    dists_to_wards = haversine_matrix(sites, stations)
    nearest_station = dists_to_wards.argmin(axis=1)
    nearest_ward_of_station = haversine_matrix(stations, sites).argmin(axis=1)

    design = []
    for s in range(stations.shape[0]):
        load = float(demand[nearest_station == s].sum())
        fast = max(1, int(np.ceil((load * 0.6) / CHARGER_FAST_KW)))
        slow = max(1, int(np.ceil((load * 0.4) / CHARGER_SLOW_KW)))
        total_kw = fast * CHARGER_FAST_KW + slow * CHARGER_SLOW_KW
        design.append({
            "station": s + 1,
            "nearest_ward": ward_names_list[int(nearest_ward_of_station[s])],
            "fast_chargers": fast,
            "slow_chargers": slow,
            "total_kw": round(total_kw, 1),
            "demand_kwh": round(load, 2),
        })
    return design

Writing backend/core.py


In [ ]:
%%writefile backend/gpu.py

"""
gpu.py
======
Real CUDA kernel for fitness evaluation, implementing the *exact same*
formula as core.compute_fitness (coverage / distance / overload — paper
Eq. 2). This matters: the previous CUDA kernel computed a completely
different quantity (mean nearest-neighbour distance only, Euclidean on
raw lat/lon degrees) than the CPU path, so any CPU-vs-GPU "fitness"
comparison in the old code was comparing two different functions, not
CPU vs GPU performance on the same objective.

This file requires an actual CUDA-capable GPU + the CUDA toolkit to run
the kernel. On a machine without a GPU, cuda_is_available() returns False
and core.run_cuda_pso() falls back to the CPU implementation with an
explicit printed warning — it will NOT silently report a fake speedup.
"""

import math
import numpy as np

try:
    from numba import cuda
    _NUMBA_OK = True
except ImportError:
    _NUMBA_OK = False

EARTH_RADIUS_KM = 6371.0
MAX_K = 32  # local-array upper bound for stations per particle on device


def cuda_is_available():
    if not _NUMBA_OK:
        return False
    try:
        return cuda.is_available()
    except Exception:
        return False


if _NUMBA_OK:

    @cuda.jit(device=True, inline=True)
    def _haversine_km(lat1, lon1, lat2, lon2):
        p1 = lat1 * math.pi / 180.0
        p2 = lat2 * math.pi / 180.0
        dphi = (lat2 - lat1) * math.pi / 180.0
        dlmb = (lon2 - lon1) * math.pi / 180.0
        a = math.sin(dphi / 2.0) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2.0) ** 2
        a = min(1.0, max(0.0, a))
        c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
        return EARTH_RADIUS_KM * c

    @cuda.jit
    def _fitness_kernel(pos, fitness, sites, demand, n_sites, k,
                         coverage_radius, grid_cap, alpha, beta, gamma, max_dist_norm):
        i = cuda.grid(1)
        if i >= pos.shape[0]:
            return

        idxs = cuda.local.array(MAX_K, dtype=numba_int32)
        for j in range(k):
            v = pos[i, j]
            idx = int(v * n_sites) % n_sites
            collision = True
            while collision:
                collision = False
                for jj in range(j):
                    if idxs[jj] == idx:
                        collision = True
                        idx = (idx + 1) % n_sites
                        break
            idxs[j] = idx

        station_load = cuda.local.array(MAX_K, dtype=numba_float64)
        for j in range(k):
            station_load[j] = 0.0

        total_demand = 0.0
        covered_demand = 0.0
        weighted_dist = 0.0

        for s in range(n_sites):
            min_d = 1.0e18
            min_j = 0
            for j in range(k):
                st = idxs[j]
                d = _haversine_km(sites[s, 0], sites[s, 1], sites[st, 0], sites[st, 1])
                if d < min_d:
                    min_d = d
                    min_j = j

            dem = demand[s]
            total_demand += dem
            weighted_dist += min_d * dem
            if min_d <= coverage_radius:
                covered_demand += dem
            station_load[min_j] += dem

        coverage = covered_demand / total_demand
        mean_dist_norm = (weighted_dist / total_demand) / max_dist_norm

        overload = 0.0
        for j in range(k):
            excess = station_load[j] - grid_cap
            if excess > 0.0:
                overload += excess
        overload = overload / (k * grid_cap)

        fitness[i] = alpha * (1.0 - coverage) + beta * mean_dist_norm + gamma * overload

    import numba
    numba_int32 = numba.int32
    numba_float64 = numba.float64


def evaluate_batch_gpu(pos, sites_arr, demand, k_stations):
    """
    pos: (n_particles, k_stations) float64 in [0,1]
    Returns fitness array (n_particles,) computed on the GPU with the kernel
    above. Only call this after checking cuda_is_available(). Used by
    verify_consistency.py for a one-shot CPU-vs-GPU fitness check; the
    actual optimization loop uses run_pso_gpu_resident below instead of
    calling this once per iteration.
    """
    from backend.core import (COVERAGE_RADIUS_KM, GRID_CAP_KW, ALPHA, BETA, GAMMA,
                               MAX_DIST_NORM_KM)

    n_particles = pos.shape[0]
    n_sites = sites_arr.shape[0]

    d_pos = cuda.to_device(np.ascontiguousarray(pos, dtype=np.float64))
    d_sites = cuda.to_device(np.ascontiguousarray(sites_arr, dtype=np.float64))
    d_demand = cuda.to_device(np.ascontiguousarray(demand, dtype=np.float64))
    d_fit = cuda.device_array(n_particles, dtype=np.float64)

    threads = 128
    blocks = (n_particles + threads - 1) // threads

    _fitness_kernel[blocks, threads](
        d_pos, d_fit, d_sites, d_demand, n_sites, k_stations,
        COVERAGE_RADIUS_KM, GRID_CAP_KW, ALPHA, BETA, GAMMA, MAX_DIST_NORM_KM,
    )
    cuda.synchronize()
    return d_fit.copy_to_host()


# =====================================================================
# GPU-RESIDENT PSO LOOP
# =====================================================================
# The previous version of this file only put the FITNESS evaluation on the
# GPU: run_cuda_pso() still did the velocity/position update in numpy on
# the CPU every iteration, and re-uploaded the entire (n_particles,
# k_stations) position matrix to the device on every single call to
# evaluate_batch_gpu. For the swarm sizes this paper actually reports on,
# that per-iteration host<->device round trip of the whole swarm is a real
# cost competing with the tiny amount of compute being offloaded — it is a
# very plausible part of why Table II's original speedup was only ~1.008
# (the paper's own Section V.B already blames "some CPU involvement in
# fitness calculation", but the update step was ALSO on the CPU, which
# that sentence didn't account for).
#
# This version keeps position, velocity, pbest, pbest_val, sites, and
# demand resident on the device for the ENTIRE run. Per iteration, only
# two small arrays cross the PCIe bus: pbest_val (n_particles floats, to
# find the argmin on the host) and — only when gbest actually improves —
# a single (k_stations,) row. The velocity/position update itself runs in
# a CUDA kernel using an on-device xoroshiro128p RNG (numba.cuda.random),
# not numpy, so no random numbers need to be generated on the host either.
if _NUMBA_OK:
    from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_uniform_float64

    @cuda.jit
    def _pso_update_kernel(pos, vel, pbest, gbest, rng_states, w, c1, c2, k):
        i = cuda.grid(1)
        if i >= pos.shape[0]:
            return
        for j in range(k):
            r1 = xoroshiro128p_uniform_float64(rng_states, i)
            r2 = xoroshiro128p_uniform_float64(rng_states, i)
            newvel = (w * vel[i, j]
                      + c1 * r1 * (pbest[i, j] - pos[i, j])
                      + c2 * r2 * (gbest[j] - pos[i, j]))
            newpos = pos[i, j] + newvel
            if newpos < 0.0:
                newpos = 0.0
            elif newpos > 1.0:
                newpos = 1.0
            vel[i, j] = newvel
            pos[i, j] = newpos

    @cuda.jit
    def _pso_pbest_update_kernel(fit, pbest_val, pos, pbest, k):
        i = cuda.grid(1)
        if i >= fit.shape[0]:
            return
        if fit[i] < pbest_val[i]:
            pbest_val[i] = fit[i]
            for j in range(k):
                pbest[i, j] = pos[i, j]


def run_pso_gpu_resident(sites_arr, demand, n_particles=50, k_stations=6, max_iter=80,
                          w_start=0.9, w_end=0.4, c1=1.5, c2=1.5, seed=None):
    """
    Fully GPU-resident PSO: update, fitness evaluation, and pbest
    maintenance all happen in CUDA kernels with pos/vel/pbest/pbest_val
    kept on the device across the whole run. Returns
    (gbest, gbest_val, history, stations) — same shape as run_cpu_pso /
    the old run_cuda_pso, so callers don't need to change.
    """
    from backend.core import (COVERAGE_RADIUS_KM, GRID_CAP_KW, ALPHA, BETA, GAMMA,
                               MAX_DIST_NORM_KM, decode_particle)

    k_stations = min(k_stations, sites_arr.shape[0])
    n_sites = sites_arr.shape[0]

    rng_np = np.random.default_rng(seed)
    pos0 = rng_np.random((n_particles, k_stations))

    d_pos = cuda.to_device(np.ascontiguousarray(pos0))
    d_vel = cuda.to_device(np.zeros_like(pos0))
    d_pbest = cuda.to_device(pos0.copy())
    d_sites = cuda.to_device(np.ascontiguousarray(sites_arr, dtype=np.float64))
    d_demand = cuda.to_device(np.ascontiguousarray(demand, dtype=np.float64))
    d_fit = cuda.device_array(n_particles, dtype=np.float64)

    threads = 128
    blocks = (n_particles + threads - 1) // threads

    seed_val = int(seed) if seed is not None else int(np.random.randint(0, 2**31 - 1))
    rng_states = create_xoroshiro128p_states(n_particles, seed=seed_val)

    # Initial fitness pass seeds pbest/pbest_val.
    _fitness_kernel[blocks, threads](
        d_pos, d_fit, d_sites, d_demand, n_sites, k_stations,
        COVERAGE_RADIUS_KM, GRID_CAP_KW, ALPHA, BETA, GAMMA, MAX_DIST_NORM_KM,
    )
    cuda.synchronize()
    d_pbest_val = cuda.to_device(d_fit.copy_to_host())

    pbest_val_host = d_pbest_val.copy_to_host()
    gbest_idx = int(np.argmin(pbest_val_host))
    gbest_val = float(pbest_val_host[gbest_idx])
    gbest_row = d_pbest[gbest_idx:gbest_idx + 1, :].copy_to_host()[0]
    d_gbest = cuda.to_device(gbest_row)

    history = [gbest_val]

    for it in range(max_iter):
        w = w_start - (w_start - w_end) * (it / max(1, max_iter - 1))

        _pso_update_kernel[blocks, threads](d_pos, d_vel, d_pbest, d_gbest, rng_states, w, c1, c2, k_stations)
        _fitness_kernel[blocks, threads](
            d_pos, d_fit, d_sites, d_demand, n_sites, k_stations,
            COVERAGE_RADIUS_KM, GRID_CAP_KW, ALPHA, BETA, GAMMA, MAX_DIST_NORM_KM,
        )
        _pso_pbest_update_kernel[blocks, threads](d_fit, d_pbest_val, d_pos, d_pbest, k_stations)
        cuda.synchronize()

        pbest_val_host = d_pbest_val.copy_to_host()  # small: n_particles floats
        it_best = int(np.argmin(pbest_val_host))
        if pbest_val_host[it_best] < gbest_val:
            gbest_val = float(pbest_val_host[it_best])
            gbest_row = d_pbest[it_best:it_best + 1, :].copy_to_host()[0]  # small: k floats
            d_gbest = cuda.to_device(gbest_row)

        history.append(gbest_val)

    gbest_final = d_gbest.copy_to_host()
    stations = decode_particle(gbest_final, sites_arr, k_stations)
    return gbest_final, gbest_val, history, stations


if __name__ == "__main__":
    print("numba installed:", _NUMBA_OK)
    print("CUDA device available:", cuda_is_available())

Writing backend/gpu.py


In [ ]:
%%writefile backend/ablation.py


"""
ablation.py
===========
Table III in the paper claims three distinct demand-modeling conditions
(uniform / demand-based / "advanced" time-series). The previous version of
this file generated V3 by calling run_cpu_pso on the *exact same* `demand`
array used for V2 — see the comment that was literally in that file:
"# V3: Same but call it advanced". V2 and V3 were mathematically guaranteed
to differ only by PSO's own randomness, not by any change in the model.

This version runs three genuinely different demand models, all defined in
core.generate_demand():
  V1 uniform  — every ward has identical demand (no spatial signal at all)
  V2 density  — demand proportional to population density (spatial only)
  V3 temporal — density scaled by the dual-peak daily profile from
                Section IV-B (spatial + temporal)

Each condition is run n_runs times (fresh random PSO seed each time, same
demand-generation seed within a condition so the demand array itself is
held fixed while the algorithm's own stochasticity is averaged out), and
we report mean +/- std, exactly like Table II does for CPU vs GPU.
"""

import os
import sys
import time
import numpy as np
import pandas as pd

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, BASE_DIR)

try:
    from backend.core import sites, generate_demand, run_cpu_pso
except ImportError:
    from core import sites, generate_demand, run_cpu_pso

N_RUNS = 5
N_PARTICLES = 60
MAX_ITER = 60
K_STATIONS = 6
DEMAND_SEED = 42  # fixed so V1/V2/V3 are compared under one consistent demand draw

RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)


def run_condition(name, demand_model):
    demand = generate_demand(seed=DEMAND_SEED, demand_model=demand_model)
    fitnesses, times = [], []
    for run in range(N_RUNS):
        np.random.seed(1000 + run)  # vary PSO stochasticity across runs
        t0 = time.time()
        _, fit, _, _ = run_cpu_pso(sites, demand, n_particles=N_PARTICLES,
                                    k_stations=K_STATIONS, max_iter=MAX_ITER)
        times.append(time.time() - t0)
        fitnesses.append(fit)
    return {
        "Version": name,
        "demand_model": demand_model,
        "Fitness_Mean": float(np.mean(fitnesses)),
        "Fitness_Std": float(np.std(fitnesses)),
        "Time_Mean_s": float(np.mean(times)),
        "Demand_Total_kWh": float(demand.sum()),
        "N_Runs": N_RUNS,
    }


if __name__ == "__main__":
    print("Running ablation study (3 genuinely distinct demand models,"
          f" {N_RUNS} runs each)...")

    rows = [
        run_condition("V1 Uniform Demand", "uniform"),
        run_condition("V2 Density-Based Demand", "density"),
        run_condition("V3 Dual-Peak Temporal Demand", "temporal"),
    ]

    for r in rows:
        print(f"  {r['Version']}: fitness = {r['Fitness_Mean']:.4f} "
              f"(+/- {r['Fitness_Std']:.4f}), time = {r['Time_Mean_s']:.2f}s")

    df = pd.DataFrame(rows)
    out_path = os.path.join(RESULTS_DIR, "ablation.csv")
    df.to_csv(out_path, index=False)
    print(f"\nAblation results saved to {out_path}")

Writing backend/ablation.py


In [ ]:
%%writefile backend/run_experiment.py

"""
run_experiment.py
==================
Every number written to results/ comes from an actual call into core.py.

CHANGES IN THIS VERSION, and why:

1. Added a Genetic Algorithm (core.run_ga) run alongside CPU PSO and CUDA
   PSO, using the SAME population size / iteration budget (N_PARTICLES /
   MAX_ITER -> n_individuals / max_gen) and the SAME fitness/decode, for a
   fair three-way comparison rather than a strawman GA. Paired t-tests
   (same seed per pair) compare PSO vs GA and GPU vs GA fitness.

2. Larger swarm-size sweep (50 -> 3200) for the speedup figure, plus a
   dataset-size sweep (10-50 wards) — reviewer #2 explicitly asked for
   scalability testing at larger problem sizes.

3. Paired t-test (CPU vs GPU fitness, and CPU vs GPU time) across the
   N_STAT_RUNS repeated runs — reviewer #2 explicitly asked for
   "statistical significance testing for runtime comparisons".

GPU disclosure (unchanged): if no CUDA device is present, run_cuda_pso()
falls back to CPU, and every output here is explicitly marked
gpu_available=False rather than presenting a fabricated speedup.
"""

import os
import sys
import time
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

try:
    from backend.core import (sites, ward_names, generate_demand, run_cpu_pso, run_cuda_pso,
                               run_ga, run_mopso, dual_peak_components, hourly_ward_demand,
                               coincident_peak_hour)
    from backend.gpu import cuda_is_available
except ImportError:
    from core import (sites, ward_names, generate_demand, run_cpu_pso, run_cuda_pso,
                       run_ga, run_mopso, dual_peak_components, hourly_ward_demand,
                       coincident_peak_hour)
    from gpu import cuda_is_available

RESULTS_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

N_PARTICLES = 100
K_STATIONS = 6
MAX_ITER = 50
N_STAT_RUNS = 10
SWARM_SIZES_FOR_SPEEDUP = [50, 100, 200, 400, 800, 1600, 3200]
DATASET_SIZES_FOR_SPEEDUP = [10, 20, 30, 40, 50]
SCALABILITY_REPS = 5

def warm_up_gpu(demand):
    """Warm-up to exclude JIT compilation from timing."""
    run_cpu_pso(sites, demand, n_particles=8, k_stations=K_STATIONS, max_iter=2)
    run_cuda_pso(sites, demand, n_particles=8, k_stations=K_STATIONS, max_iter=2)
    run_ga(sites, demand, n_individuals=8, k_stations=K_STATIONS, max_gen=2)


def path(name):
    return os.path.join(RESULTS_DIR, name)


def main():
    gpu_ok = cuda_is_available()
    print(f"CUDA device available: {gpu_ok}")
    if not gpu_ok:
        print("WARNING: no GPU detected. CPU-vs-GPU timing numbers below will be "
              "CPU-vs-CPU (speedup ~1x by construction) and are marked "
              "gpu_available=False in the output CSVs.")

    demand = generate_demand(seed=42, demand_model="temporal")
    warm_up_gpu(demand)
    pd.DataFrame({"ward": ward_names, "demand_kwh": demand}).to_csv(
        path("table1_demand.csv"), index=False)

    # Wards table (backs fig_map — no figure should import core.py directly;
    # every plotted number should trace back to a CSV this script wrote).
    pd.DataFrame({
        "ward": ward_names, "lat": sites[:, 0], "lon": sites[:, 1],
    }).to_csv(path("table0_wards.csv"), index=False)

    # Morning/evening components at the coincident peak hour (backs
    # fig_dual_peak). Computed once here instead of being recomputed live
    # inside graphs.py, so the figure and this CSV can never drift apart.
    morning_component, evening_component = dual_peak_components()
    peak_hour = coincident_peak_hour()
    pd.DataFrame({
        "ward": ward_names, "morning_kwh": morning_component, "evening_kwh": evening_component,
    }).to_csv(path("table1b_dual_peak.csv"), index=False)

    # City-wide hourly demand curve (backs fig_hourly_curve).
    hourly = hourly_ward_demand()
    citywide_hourly = hourly.sum(axis=0)
    pd.DataFrame({
        "hour": range(24), "citywide_kwh": citywide_hourly,
        "is_coincident_peak": [h == peak_hour for h in range(24)],
    }).to_csv(path("table1c_hourly_citywide.csv"), index=False)

    # ---------------------------------------------------------------
    # Table II — CPU-PSO vs GPU-PSO vs GA, same budget, paired significance
    # ---------------------------------------------------------------
    cpu_fits, gpu_fits, ga_fits = [], [], []
    cpu_times, gpu_times, ga_times = [], [], []
    cpu_hist_all, gpu_hist_all, ga_hist_all = [], [], []

    for run in range(N_STAT_RUNS):
        np.random.seed(2000 + run)
        t0 = time.time()
        _, fit_c, hist_c, _ = run_cpu_pso(sites, demand, N_PARTICLES, K_STATIONS, MAX_ITER)
        cpu_times.append(time.time() - t0)
        cpu_fits.append(fit_c)
        cpu_hist_all.append(hist_c)

        np.random.seed(2000 + run)  # same seed -> same initial swarm -> valid paired comparison
        t0 = time.time()
        _, fit_g, hist_g, _ = run_cuda_pso(sites, demand, N_PARTICLES, K_STATIONS, MAX_ITER)
        gpu_times.append(time.time() - t0)
        gpu_fits.append(fit_g)
        gpu_hist_all.append(hist_g)

        np.random.seed(2000 + run)  # same seed, same population/iteration budget as PSO
        t0 = time.time()
        _, fit_a, hist_a, _ = run_ga(sites, demand, n_individuals=N_PARTICLES,
                                      k_stations=K_STATIONS, max_gen=MAX_ITER)
        ga_times.append(time.time() - t0)
        ga_fits.append(fit_a)
        ga_hist_all.append(hist_a)

    total_evals = N_PARTICLES * (MAX_ITER + 1)  # same for CPU-PSO, GPU-PSO, and GA — verified
                                                  # empirically (see fairness note below) rather
                                                  # than just asserted: run_cpu_pso and run_ga
                                                  # both call compute_fitness exactly
                                                  # n_particles*(max_iter+1) times for identical
                                                  # (n_particles=n_individuals, max_iter=max_gen).

    pso_gpu_fit_t = stats.ttest_rel(cpu_fits, gpu_fits)
    pso_gpu_time_t = stats.ttest_rel(cpu_times, gpu_times)
    pso_ga_fit_t = stats.ttest_rel(cpu_fits, ga_fits)
    gpu_ga_fit_t = stats.ttest_rel(gpu_fits, ga_fits)

    summary = pd.DataFrame({
        "Metric": ["CPU-PSO Mean", "GPU-PSO Mean", "GA Mean",
                   "CPU-PSO Std", "GPU-PSO Std", "GA Std",
                   "CPU-PSO Time", "GPU-PSO Time", "GA Time",
                   "CPU-PSO Time_per_eval_ms", "GA Time_per_eval_ms",
                   "Total_Fitness_Evaluations (all three, identical)",
                   "Speedup (PSO CPU/GPU)",
                   "Fitness_paired_t_pvalue (CPU-PSO vs GPU-PSO)",
                   "Time_paired_t_pvalue (CPU-PSO vs GPU-PSO)",
                   "Fitness_paired_t_pvalue (CPU-PSO vs GA)",
                   "Fitness_paired_t_pvalue (GPU-PSO vs GA)"],
        "Value": [
            np.mean(cpu_fits), np.mean(gpu_fits), np.mean(ga_fits),
            np.std(cpu_fits), np.std(gpu_fits), np.std(ga_fits),
            np.mean(cpu_times), np.mean(gpu_times), np.mean(ga_times),
            1000 * np.mean(cpu_times) / total_evals,
            1000 * np.mean(ga_times) / total_evals,
            total_evals,
            np.mean(cpu_times) / np.mean(gpu_times) if np.mean(gpu_times) > 0 else float("nan"),
            pso_gpu_fit_t.pvalue, pso_gpu_time_t.pvalue,
            pso_ga_fit_t.pvalue, gpu_ga_fit_t.pvalue,
        ],
    })
    summary["gpu_available"] = gpu_ok
    summary["n_runs"] = N_STAT_RUNS
    summary.to_csv(path("table2_performance_summary.csv"), index=False)
    print("\nTable II (Performance Summary — CPU-PSO vs GPU-PSO vs GA):")
    print(summary.to_string(index=False))

    # ---------------------------------------------------------------
    # Fig 2 — convergence curves (mean across runs), now with GA too
    # ---------------------------------------------------------------
    min_len = min(min(len(h) for h in cpu_hist_all), min(len(h) for h in ga_hist_all))
    cpu_curve = np.mean([h[:min_len] for h in cpu_hist_all], axis=0)
    gpu_curve = np.mean([h[:min_len] for h in gpu_hist_all], axis=0)
    ga_curve = np.mean([h[:min_len] for h in ga_hist_all], axis=0)
    pd.DataFrame({
        "iteration": range(min_len),
        "cpu_fitness": cpu_curve,
        "gpu_fitness": gpu_curve,
        "ga_fitness": ga_curve,
    }).to_csv(path("table_convergence.csv"), index=False)

    # ---------------------------------------------------------------
    # Fig 3 — speedup vs swarm size (up to 3200 particles)
    # ---------------------------------------------------------------
    speedup_rows = []
    for n in SWARM_SIZES_FOR_SPEEDUP:
        cpu_rep=[]
        gpu_rep=[]

        for rep in range(SCALABILITY_REPS):

            np.random.seed(3000+rep)
            t0=time.time()
            run_cpu_pso(sites,demand,n,K_STATIONS,MAX_ITER)
            cpu_rep.append(time.time()-t0)

            np.random.seed(3000+rep)
            t0=time.time()
            run_cuda_pso(sites,demand,n,K_STATIONS,MAX_ITER)
            gpu_rep.append(time.time()-t0)

        t_cpu=np.mean(cpu_rep)
        t_gpu=np.mean(gpu_rep)
        t_cpu_sd=np.std(cpu_rep)
        t_gpu_sd=np.std(gpu_rep)

        speedup_rows.append({
            "n_particles":n,
            "cpu_time":t_cpu,
            "cpu_time_std":t_cpu_sd,
            "gpu_time":t_gpu,
            "gpu_time_std":t_gpu_sd,
            "speedup":t_cpu/t_gpu,
            "gpu_available":gpu_ok,
            "n_reps":SCALABILITY_REPS
        })
        print(f"  swarm size {n}: CPU {t_cpu:.3f}s, GPU/fallback {t_gpu:.3f}s")
    pd.DataFrame(speedup_rows).to_csv(path("table_speedup_vs_particles.csv"), index=False)

    # ---------------------------------------------------------------
    # Fig 3b — speedup vs dataset size (n_sites), fixed swarm size
    # ---------------------------------------------------------------
    dataset_rows = []
    full_demand = demand
    for n_sites_test in DATASET_SIZES_FOR_SPEEDUP:
        sub_sites = sites[:n_sites_test]
        sub_demand = full_demand[:n_sites_test]
        k_test = min(K_STATIONS, n_sites_test)

        cpu_rep=[]
        gpu_rep=[]

        for rep in range(SCALABILITY_REPS):

            np.random.seed(4000+rep)
            t0=time.time()
            run_cpu_pso(sub_sites,sub_demand,N_PARTICLES,k_test,MAX_ITER)
            cpu_rep.append(time.time()-t0)

            np.random.seed(4000+rep)
            t0=time.time()
            run_cuda_pso(sub_sites,sub_demand,N_PARTICLES,k_test,MAX_ITER)
            gpu_rep.append(time.time()-t0)

        t_cpu=np.mean(cpu_rep)
        t_gpu=np.mean(gpu_rep)
        t_cpu_sd=np.std(cpu_rep)
        t_gpu_sd=np.std(gpu_rep)

        dataset_rows.append({
            "n_sites":n_sites_test,
            "cpu_time":t_cpu,
            "cpu_time_std":t_cpu_sd,
            "gpu_time":t_gpu,
            "gpu_time_std":t_gpu_sd,
            "speedup":t_cpu/t_gpu,
            "gpu_available":gpu_ok,
            "n_reps":SCALABILITY_REPS
        })
    pd.DataFrame(dataset_rows).to_csv(path("table_speedup_vs_dataset_size.csv"), index=False)

    # ---------------------------------------------------------------
    # Fig 4 — Pareto front (adaptive-grid archive-based MOPSO)
    # ---------------------------------------------------------------
    np.random.seed(42)
    archive_pos, archive_obj = run_mopso(sites, demand, n_particles=60, max_iter=60,
                                          archive_max_size=200, n_restarts=8)

    # Defensive re-verification: run_mopso's internal archive already
    # enforces non-domination and de-duplication on every insert, but this
    # re-checks the FINAL returned set independently, right before writing
    # the CSV a reviewer might inspect directly — so a bug anywhere in the
    # archive's insert/trim logic can never silently leak a dominated or
    # duplicate point into the figure.
    def _is_dominated_by_any(i, obj):
        for j in range(len(obj)):
            if j == i:
                continue
            if (obj[j][0] <= obj[i][0] and obj[j][1] <= obj[i][1] and
                    (obj[j][0] < obj[i][0] or obj[j][1] < obj[i][1])):
                return True
        return False

    keep_mask = np.array([not _is_dominated_by_any(i, archive_obj) for i in range(len(archive_obj))])
    clean_obj = archive_obj[keep_mask]
    _, unique_idx = np.unique(clean_obj.round(9), axis=0, return_index=True)
    clean_obj = clean_obj[np.sort(unique_idx)]

    n_removed = len(archive_obj) - len(clean_obj)
    if n_removed > 0:
        print(f"WARNING: removed {n_removed} dominated/duplicate point(s) from the "
              f"MOPSO archive that should not have been there — this indicates a bug "
              f"in the archive's insert logic and should be investigated, not just "
              f"silently patched over release after release.")

    pd.DataFrame(clean_obj, columns=["cost", "coverage_loss"]).to_csv(
        path("table_pareto.csv"), index=False)
    print(f"\nMOPSO archive size: {len(clean_obj)} non-dominated solutions "
          f"({n_removed} removed by the final verification pass)")

    print(f"\nAll results written to: {RESULTS_DIR}")


if __name__ == "__main__":
    main()

Writing backend/run_experiment.py


In [ ]:
%%writefile backend/graphs.py

"""
graphs.py
=========
Every figure here is plotted from a CSV that run_experiment.py / ablation.py
actually wrote — this file makes ZERO calls into core.py. That's a
deliberate change from an earlier version, where fig_dual_peak,
fig_hourly_curve, and fig_map imported core.py and recomputed values live
at plot time. Those numbers were already correct (same deterministic
functions), but "correct" isn't the same guarantee as "traceable to a
saved artifact a reviewer can open and check independently" — reading only
from CSVs that run_experiment.py wrote means every figure has an audit
trail, and a figure and its underlying data can never silently drift apart
from each other.

CHANGES IN THIS VERSION, and why:

1. fig_convergence's title/legend are now built FROM the CSV's own columns
   instead of being hardcoded text, so the title can never claim a curve
   that isn't actually plotted (or vice versa).
2. fig_pareto now uses a bare scatter plot — no connecting line. Points in
   a Pareto archive are discrete solutions; drawing a line through them
   visually implies a continuum of achievable trade-offs between them,
   which isn't true (nothing was evaluated in the gaps).
3. fig_dual_peak, fig_hourly_curve, fig_map now read table1b_dual_peak.csv,
   table1c_hourly_citywide.csv, table0_wards.csv respectively, instead of
   importing core.py.
4. fig_speedup / fig_speedup_vs_dataset_size now read the gpu_available
   column those CSVs carry and put an explicit "(CPU fallback — no GPU
   detected)" qualifier directly in the figure title when it's False, so
   the figure is honest about what it shows without requiring the reader
   to cross-reference Table II or console output.

Run backend/run_experiment.py and backend/ablation.py first to produce the
CSVs this script reads.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS_DIR = os.path.join(BASE_DIR, "results")


def rpath(name):
    return os.path.join(RESULTS_DIR, name)


def savefig(name):
    plt.tight_layout()
    plt.savefig(rpath(name), dpi=150)
    plt.close()
    print(f"  wrote {name}")


def fig_demand():
    df = pd.read_csv(rpath("table1_demand.csv"))
    plt.figure(figsize=(7, 4))
    plt.bar(df["ward"], df["demand_kwh"])
    plt.xticks(rotation=75, ha="right", fontsize=7)
    plt.ylabel("Demand (kWh)")
    plt.title("Ward-level EV Charging Demand (temporal model, coincident peak hour)")
    savefig("fig1_demand.png")


def fig_dual_peak():
    df = pd.read_csv(rpath("table1b_dual_peak.csv"))
    order = (-df[["morning_kwh", "evening_kwh"]].max(axis=1)).argsort().values
    plt.figure(figsize=(8, 4))
    x = range(len(df))
    plt.bar([i - 0.2 for i in x], df["morning_kwh"].values[order], width=0.4,
            label="Morning component (commercial)", color="#e8ff47")
    plt.bar([i + 0.2 for i in x], df["evening_kwh"].values[order], width=0.4,
            label="Evening component (residential)", color="#4c9eff")
    plt.xticks(list(x), df["ward"].values[order], rotation=80, ha="right", fontsize=6)
    plt.ylabel("Demand (kWh)")
    plt.title("Morning vs Evening Demand Components at Coincident Peak Hour")
    plt.legend()
    savefig("fig1b_dual_peak_demand.png")


def fig_hourly_curve():
    df = pd.read_csv(rpath("table1c_hourly_citywide.csv"))
    peak_row = df[df["is_coincident_peak"]]
    peak_hour = int(peak_row["hour"].iloc[0]) if len(peak_row) else None
    plt.figure(figsize=(7, 4))
    plt.plot(df["hour"], df["citywide_kwh"], marker="o", color="#ff6b35")
    if peak_hour is not None:
        plt.axvline(peak_hour, color="gray", linestyle="--", linewidth=1,
                    label=f"Coincident peak hour ({peak_hour}:00)")
        plt.legend()
    plt.xlabel("Hour of day")
    plt.ylabel("City-wide demand (kWh)")
    plt.title("City-wide Hourly EV Charging Demand (dual-peak model)")
    savefig("fig1c_hourly_citywide_demand.png")


def fig_convergence():
    df = pd.read_csv(rpath("table_convergence.csv"))
    # Column -> (label, color). Only columns actually present get plotted
    # AND named in the title/legend — this mapping is the single source of
    # truth for both, so they cannot disagree with each other.
    series_spec = {
        "cpu_fitness": ("CPU PSO", "#4c9eff"),
        "gpu_fitness": ("CUDA PSO", "#ff6b35"),
        "ga_fitness": ("GA", "#a259ff"),
    }
    present = [col for col in series_spec if col in df.columns]

    plt.figure(figsize=(6, 4))
    for col in present:
        label, color = series_spec[col]
        plt.plot(df["iteration"], df[col], label=label, color=color)
    plt.xlabel("Iteration / Generation")
    plt.ylabel("Fitness")
    plt.title("Convergence Curve — " + " vs ".join(series_spec[c][0] for c in present))
    plt.legend()
    savefig("fig2_convergence.png")


def _gpu_suffix(df):
    if "gpu_available" in df.columns and not bool(df["gpu_available"].iloc[0]):
        return " (CPU fallback — no GPU detected)"
    return ""


def fig_speedup():
    df = pd.read_csv(rpath("table_speedup_vs_particles.csv"))
    plt.figure(figsize=(6, 4))
    plt.plot(df["n_particles"], df["speedup"], marker="o", color="#e8ff47")
    plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
    plt.xlabel("Number of Particles")
    plt.ylabel("Speedup (CPU time / GPU time)")
    plt.title("Speedup vs Swarm Size" + _gpu_suffix(df))
    savefig("fig3_speedup.png")


def fig_speedup_vs_dataset_size():
    df = pd.read_csv(rpath("table_speedup_vs_dataset_size.csv"))
    plt.figure(figsize=(6, 4))
    plt.plot(df["n_sites"], df["speedup"], marker="o", color="#ff6b35")
    plt.axhline(1.0, color="gray", linestyle="--", linewidth=1)
    plt.xlabel("Number of Wards (dataset size)")
    plt.ylabel("Speedup (CPU time / GPU time)")
    plt.title("Speedup vs Dataset Size" + _gpu_suffix(df))
    savefig("fig3b_speedup_vs_dataset_size.png")


def fig_pareto():
    df = pd.read_csv(rpath("table_pareto.csv"))
    plt.figure(figsize=(6, 4))
    # Scatter only — NO connecting line. Each point is an independently
    # evaluated, non-dominated station layout; nothing was evaluated
    # between them, so a line would imply a continuum that doesn't exist.
    plt.scatter(df["cost"], df["coverage_loss"], color="#36d399", s=45, zorder=3)
    plt.xlabel("Cost (infrastructure + distance + overload)")
    plt.ylabel("Coverage Loss")
    plt.title(f"Pareto Archive — Non-Dominated Solutions (N={len(df)})")
    savefig("fig4_pareto.png")


def fig_map():
    df = pd.read_csv(rpath("table0_wards.csv"))
    plt.figure(figsize=(7, 6))
    plt.scatter(df["lon"], df["lat"], c="#4c9eff", s=25)
    for _, row in df.iterrows():
        plt.annotate(row["ward"], (row["lon"], row["lat"]), fontsize=5, alpha=0.8)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"BBMP Ward Locations")
    savefig("fig5_map.png")


def fig_ablation():
    df = pd.read_csv(rpath("ablation.csv"))
    n_runs = int(df["N_Runs"].iloc[0]) if "N_Runs" in df.columns else None
    plt.figure(figsize=(6, 4))
    plt.bar(df["Version"], df["Fitness_Mean"], yerr=df["Fitness_Std"], capsize=4,
            color=["#4c9eff", "#e8ff47", "#ff6b35"])
    plt.ylabel("Fitness (lower = better)")
    title = "Ablation: Demand Model Comparison"
    if n_runs:
        title += f" (mean ± std, N={n_runs} runs)"
    plt.title(title)
    plt.xticks(rotation=15, ha="right")
    savefig("fig6_ablation.png")


if __name__ == "__main__":
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print("Generating figures from computed results (no synthetic data, no live "
          "core.py calls — every number below is read from a CSV)...")
    fig_demand()
    fig_dual_peak()
    fig_hourly_curve()
    fig_convergence()
    fig_speedup()
    fig_speedup_vs_dataset_size()
    fig_pareto()
    fig_map()
    fig_ablation()
    print("Done.")

Writing backend/graphs.py


In [ ]:
%%writefile backend/verify_consistency.py
import os
import sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

try:
    from backend.core import sites, generate_demand, compute_fitness, decode_particle
    from backend.gpu import cuda_is_available, evaluate_batch_gpu
except ImportError:
    from core import sites, generate_demand, compute_fitness, decode_particle
    from gpu import cuda_is_available, evaluate_batch_gpu


def main():
    if not cuda_is_available():
        print("No CUDA device detected — cannot verify the GPU kernel here.")
        return

    np.random.seed(123)
    demand = generate_demand(seed=123)
    k_stations = 6
    n_particles = 500

    pos = np.random.rand(n_particles, k_stations)

    cpu_fit = np.array([
        compute_fitness(decode_particle(pos[i], sites, k_stations), sites, demand)
        for i in range(n_particles)
    ])
    gpu_fit = evaluate_batch_gpu(pos, sites, demand, k_stations)

    abs_diff = np.abs(cpu_fit - gpu_fit)
    print(f"n_particles checked: {n_particles}")
    print(f"max abs difference:  {abs_diff.max():.3e}")
    print(f"mean abs difference: {abs_diff.mean():.3e}")

    tol = 1e-6
    if abs_diff.max() < tol:
        print(f"PASS: CPU and GPU fitness match within {tol:.0e} for all {n_particles} particles.")
    else:
        print(f"FAIL: CPU and GPU fitness differ by more than {tol:.0e}.")


if __name__ == "__main__":
    main()

Writing backend/verify_consistency.py


In [ ]:
!python backend/verify_consistency.py

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
n_particles checked: 500
max abs difference:  7.827e-15
mean abs difference: 1.729e-15
PASS: CPU and GPU fitness match within 1e-06 for all 500 particles.


In [ ]:
!python backend/run_experiment.py
!python backend/ablation.py
!python backend/graphs.py

CUDA device available: True
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))

Table II (Performance Summary — CPU-PSO vs GPU-PSO vs GA):
                                          Metric       Value  gpu_available  n_runs
                                    CPU-PSO Mean    0.137086           True      10
                                    GPU-PSO Mean    0.135878

In [ ]:
from google.colab import files
!zip -r results.zip results/
files.download("results.zip")

  adding: results/ (stored 0%)
  adding: results/table2_performance_summary.csv (deflated 54%)
  adding: results/table_speedup_vs_dataset_size.csv (deflated 46%)
  adding: results/fig1_demand.png (deflated 13%)
  adding: results/ablation.csv (deflated 34%)
  adding: results/fig3_speedup.png (deflated 11%)
  adding: results/table_speedup_vs_particles.csv (deflated 46%)
  adding: results/table_pareto.csv (deflated 46%)
  adding: results/table1_demand.csv (deflated 39%)
  adding: results/table_convergence.csv (deflated 62%)
  adding: results/fig3b_speedup_vs_dataset_size.png (deflated 12%)
  adding: results/fig1c_hourly_citywide_demand.png (deflated 8%)
  adding: results/fig1b_dual_peak_demand.png (deflated 13%)
  adding: results/fig6_ablation.png (deflated 14%)
  adding: results/fig4_pareto.png (deflated 16%)
  adding: results/table1b_dual_peak.csv (deflated 41%)
  adding: results/table0_wards.csv (deflated 48%)
  adding: results/fig2_convergence.png (deflated 7%)
  adding: results/table

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>